# DV1597 - Assignment 2

### Maximilian Carlbäck - maco24@student.bth.se

### Adam Block - ...@student.bth.se

Initialize data

In [ ]:
import pandas as pd
import datetime

df_C19_daily_number_cases = pd.read_csv('1.COVID-19_daily_number_of_new_cases_and_deaths.csv')

df_C19_vaccination = pd.read_csv('2.COVID-19_vaccination.csv')

df_C19_hospital_ICU = pd.read_csv('3.COVID-19_hospital_and_ICU_admission_rates.csv')

pd.set_option('display.max_columns', None)

# print(df_C19_daily_number_cases.head())
# print(df_C19_vaccination.head())
# print(df_C19_hospital_ICU.head())

In [ ]:
print(len(df_C19_daily_number_cases))
df_C19_daily_number_cases_nodupes = df_C19_daily_number_cases.drop_duplicates()
print(len(df_C19_daily_number_cases_nodupes))

print(len(df_C19_vaccination))
df_C19_vaccination_nodupes = df_C19_vaccination.drop_duplicates()
print(len(df_C19_vaccination_nodupes))

print(len(df_C19_hospital_ICU))
df_C19_hospital_ICU_nodupes = df_C19_hospital_ICU.drop_duplicates()
print(len(df_C19_hospital_ICU_nodupes))

In [ ]:
df_C19_daily_number_cases["dateRep"] = pd.to_datetime(df_C19_daily_number_cases["dateRep"])
neg = df_C19_daily_number_cases[df_C19_daily_number_cases["cases"]<0] 
neg
#convert to datetime

In [ ]:

if (df_C19_daily_number_cases["deaths"].isnull().any()):
    df_C19_daily_number_cases["deaths"] = 0
    
print(df_C19_daily_number_cases["deaths"].isnull().unique())


Vi antar att deaths är 0 när det är NaN värde för att de kringliggande värdena är nära 0

In [ ]:

df_C19_daily_number_cases["deaths"] = df_C19_daily_number_cases["deaths"].fillna(0)


Vi såg att spanien bara rapporterade var tredje dag (isch) så vi satte alla NaN värden i "cases" kolumnen till 0

In [ ]:
df_C19_daily_number_cases.loc[
    (df_C19_daily_number_cases["geoId"] == "ES") &
    (df_C19_daily_number_cases["cases"].isna()),
    "cases"
] = 0

print(df_C19_daily_number_cases[(df_C19_daily_number_cases["geoId"] == "ES") & (df_C19_daily_number_cases["cases"].isna())])

Vi såg också att NaN värdena på "cases" på Island verkade inte vara borttappade värden utan snarare ackummulerade, då efter dagar av NaN värde kom ett mycket större värde ochs sedan tillbaka till det "normala". Vi satte också alla dessa värden till 0

In [ ]:
df_C19_daily_number_cases.loc[
    (df_C19_daily_number_cases["geoId"] == "IS") &
    (df_C19_daily_number_cases["cases"].isna()),
    "cases"
] = 0

print(df_C19_daily_number_cases[(df_C19_daily_number_cases["geoId"] == "IS") & (df_C19_daily_number_cases["cases"].isna())])

Detsamma som Island gäller Danmark

In [ ]:
df_C19_daily_number_cases.loc[
    (df_C19_daily_number_cases["geoId"] == "DK") &
    (df_C19_daily_number_cases["cases"].isna()),
    "cases"
] = 0

print(df_C19_daily_number_cases[(df_C19_daily_number_cases["geoId"] == "DK") & (df_C19_daily_number_cases["cases"].isna())])

Vi sätter Estonia och Norway till 0 också

In [ ]:
df_C19_daily_number_cases.loc[
    (df_C19_daily_number_cases["geoId"] == "EE") &
    (df_C19_daily_number_cases["cases"].isna()),
    "cases"
] = 0

print(df_C19_daily_number_cases[(df_C19_daily_number_cases["geoId"] == "EE") & (df_C19_daily_number_cases["cases"].isna())])

df_C19_daily_number_cases.loc[
    (df_C19_daily_number_cases["geoId"] == "NO") &
    (df_C19_daily_number_cases["cases"].isna()),
    "cases"
] = 0 

print(df_C19_daily_number_cases[(df_C19_daily_number_cases["geoId"] == "NO") & (df_C19_daily_number_cases["cases"].isna())])

In [ ]:
print(df_C19_daily_number_cases[df_C19_daily_number_cases["cases"].isnull()])
# print(df_C19_daily_number_cases[df_C19_daily_number_cases["geoId"] == "DK"])

In [ ]:
print(len(df_C19_daily_number_cases["geoId"].unique()) == len(df_C19_daily_number_cases["popData2020"].unique())
)


In [ ]:
nations = df_C19_daily_number_cases["geoId"].unique()
print( nations)

print(df_C19_daily_number_cases["dateRep"].max())
df_C19_daily_number_cases["dateRep"].min()

In [ ]:
scoreboard = pd.DataFrame({"quarter":[],"geoId":[],"cases":[]})
startTime = pd.to_datetime("2020-01-01")
endTime = pd.to_datetime("2022-10-26")

quartile = 1
threshold = startTime + pd.DateOffset(months=3)

#gå igenom alla datum
while startTime<endTime:
   df_quarterly = df_C19_daily_number_cases[(df_C19_daily_number_cases["dateRep"] >= startTime ) & (df_C19_daily_number_cases["dateRep"] < threshold)].copy()
   df_quarterly = df_quarterly.groupby("geoId")["cases"].sum().reset_index() #is now a series
   
  
   df_quarterly.sort_values(by="cases",inplace=True,ascending=False)
   top10 = df_quarterly.iloc[0:10]
   
   scoreboard.loc[len(scoreboard)] = [f"Q{quartile}", top10.iloc[0]["geoId"],top10.iloc[0]["cases"]]

   startTime = threshold
   threshold = threshold + pd.DateOffset(months=3)
   quartile+=1 

print(scoreboard)

In [ ]:
import plotly.express as px

# Make sure dateRep is datetime
df_C19_daily_number_cases["dateRep"] = pd.to_datetime(df_C19_daily_number_cases["dateRep"], dayfirst=True, errors="coerce")

# Create year column
df_C19_daily_number_cases["year"] = df_C19_daily_number_cases["dateRep"].dt.year

# Keep only 2020, 2021 and 2022
df_filtered = df_C19_daily_number_cases[df_C19_daily_number_cases["year"].isin([2020, 2021, 2022])]

# Group by country and year
df_country_year = df_filtered.groupby(["year", "countriesAndTerritories", "countryterritoryCode"],as_index=False)[["cases", "deaths"]].sum()

fig_cases = px.choropleth(
    df_country_year,
    locations="countryterritoryCode",
    color="cases",
    hover_name="countriesAndTerritories",
    animation_frame="year",
    range_color=(0, max(df_country_year["cases"])),
    color_continuous_scale=px.colors.sequential.YlOrRd,
    title="Total COVID-19 Cases by Country in 2020, 2021 and 2022",
    width=800,
    height=600
)

fig_cases.update_layout(
    geo=dict(scope="europe", resolution=50)
)

fig_cases.show()

fig_deaths = px.choropleth(
    df_country_year,
    locations="countryterritoryCode",
    color="deaths",
    hover_name="countriesAndTerritories",
    animation_frame="year",
    range_color=(0, max(df_country_year["deaths"])),
    color_continuous_scale=px.colors.sequential.Reds,
    title="Total COVID-19 Deaths by Country in 2020, 2021 and 2022",
    width=800,
    height=600
)

fig_deaths.update_layout(
    geo=dict(scope="europe", resolution=50)
)

fig_deaths.show()

# Cleaning vaccination data

In [ ]:
print(df_C19_vaccination[df_C19_vaccination["Vaccine"].isnull()])

print(df_C19_vaccination[df_C19_vaccination["TargetGroup"].isna()])

print(df_C19_vaccination[df_C19_vaccination["ReportingCountry"].isna()])

print(df_C19_vaccination[df_C19_vaccination["NumberDosesReceived"].isna()])

print(df_C19_vaccination[df_C19_vaccination["Denominator"].isna()])

In [ ]:
df_C19_vaccination["FirstDoseRefused"] = df_C19_vaccination["FirstDoseRefused"].fillna(0)

print(df_C19_vaccination[df_C19_vaccination["FirstDoseRefused"].isna()]) 

df_C19_vaccination["NumberDosesReceived"] = df_C19_vaccination["NumberDosesReceived"].fillna(0)

print(df_C19_vaccination[df_C19_vaccination["NumberDosesReceived"].isna()]) 



# 3.3

In [ ]:
#print(df_C19_vaccination[df_C19_vaccination["Region"] != df_C19_vaccination["ReportingCountry"]])
c = df_C19_vaccination["ReportingCountry"].unique()

for x in c:
    cntry = df_C19_vaccination[df_C19_vaccination["ReportingCountry"]==x]
    print(x, cntry["TargetGroup"].unique(),"\n")

    print(x,cntry["Vaccine"].unique(),"\n")

AT ['Age10_14' 'Age<18' 'Age60_69' 'Age15_17' 'ALL' 'Age5_9' 'Age18_24'
 'Age0_4' 'Age70_79' 'Age80+' 'Age25_49' 'Age50_59'] 

BE ['HCW' 'AgeUNK' 'Age80+' 'Age70_79' 'Age60_69' 'Age50_59' 'Age25_49'
 'Age<18' 'LTCF' 'ALL' 'Age18_24'] 

BG ['Age25_49' 'Age50_59' 'Age60_69' 'Age70_79' 'Age80+' 'AgeUNK' 'Age<18'
 'ALL' 'Age18_24' 'HCW' 'LTCF'] 

CY ['Age15_17' 'Age18_24' 'Age25_49' 'Age50_59' 'Age60_69' 'Age70_79'
 'Age80+' 'ALL' 'HCW' 'LTCF' 'Age5_9' 'Age10_14' 'Age0_4' 'AgeUNK'] 

CZ ['Age50_59' 'Age18_24' 'Age15_17' 'Age10_14' 'Age<18' 'LTCF' 'HCW' 'ALL'
 'Age80+' 'Age70_79' 'Age60_69' 'Age25_49' 'Age5_9' 'Age0_4'] 

DE ['1_Age60+' '1_Age<60' 'ALL' 'Age<18'] 

DK ['ALL' 'Age80+' 'HCW' 'LTCF' 'Age10_14' 'Age15_17' 'Age18_24' 'Age25_49'
 'Age50_59' 'Age60_69' 'Age5_9' 'Age70_79' 'Age0_4'] 

EE ['Age80+' 'Age70_79' 'Age60_69' 'Age<18' 'ALL' 'AgeUNK' 'Age50_59'
 'Age25_49' 'Age18_24' 'LTCF' 'HCW'] 

EL ['Age18_24' 'Age15_17' 'Age10_14' 'Age5_9' 'Age0_4' 'Age<18' 'Age80+'
 'Age70_79' 'Age60

In [ ]:
# Calculate total doses received for each vaccine and get the top 3


dose_columns = ["FirstDose", "SecondDose", "DoseAdditional1", "DoseAdditional2", "DoseAdditional3", "DoseAdditional4", "DoseAdditional5", "UnknownDose"]

df_filtered = df_C19_vaccination[df_C19_vaccination["TargetGroup"].isin(["ALL", "Age<18"])]
df_filtered = df_filtered[df_filtered["Region"] == df_filtered["ReportingCountry"]]
df_filtered = df_filtered[df_filtered["Vaccine"] != "UNK"]

# Calculate the total doses received for each vaccine for each country

df_filtered["TotalDoses"] = df_filtered[dose_columns].sum(axis=1)

print(df_filtered["TotalDoses"].sum())

top_3_eu_vaccines = (
    df_filtered
    .groupby("Vaccine")["TotalDoses"]
    .sum()
    .reset_index()
    .sort_values("TotalDoses", ascending=False)
    .head(3)
)

print(top_3_eu_vaccines.to_string(index=False))

# WIP

top_vaccine_per_country = (
    df_filtered
    .groupby(["ReportingCountry", "Vaccine"])["TotalDoses"]
    .sum()
    .reset_index()
    .sort_values(["ReportingCountry", "TotalDoses"], ascending=[True, False])
    .groupby("ReportingCountry")
    .head(3)
)

print(top_vaccine_per_country.to_string(index=False))

The most popular vaccines are COM, MOD, and AZ. It seems like every countries most used vaccines are one of the three top vaccines across Europe. FOr most countries pfiser, moderna and AztraZeneca are the most popular, with some exeptions like Janssen and Beijing CNBG. 



# 3.4

In [ ]:
print(df_filtered)

In [ ]:
testframe=df_C19_vaccination[df_C19_vaccination["ReportingCountry"]=="FR"]
print(testframe[testframe["Vaccine"]=="COM"])

In [ ]:
#Germany will not be counted as it only lists UNK
vaxlist = ["COM","MOD","AZ"]
df_task4 = df_C19_vaccination[df_C19_vaccination["Region"] == df_C19_vaccination["ReportingCountry"]].copy()
df_task4 = df_task4[df_task4["Vaccine"] != "UNK"]
df_task4["TotalDoses"] = df_task4[dose_columns].sum(axis=1)


groupNcountrydf =pd.DataFrame({"country":[],"vaccine":[],"ageGroup":[]})

nations = df_task4["ReportingCountry"].unique()

for nation in nations:
    mostPopular=df_task4[df_task4["ReportingCountry"]==nation]
    for vax in vaxlist:
    

        mostPopular2=mostPopular[mostPopular["Vaccine"]==vax].copy()

        if not mostPopular2.empty:

        
            perGroup = mostPopular2.groupby("TargetGroup")["TotalDoses"].sum().reset_index()
            perGroup.sort_values(by="TotalDoses",inplace=True,ascending=False)
        
            try:
          
                ageGroup = (perGroup["TargetGroup"].iloc[1])
            except:
        
                ageGroup = (perGroup["TargetGroup"].loc[0])
            groupNcountrydf.loc[len(groupNcountrydf)] = [nation,vax,ageGroup]

print(groupNcountrydf)
     




# 3.5

In [ ]:
print(df_C19_vaccination)

In [ ]:
nations = df_C19_vaccination["ReportingCountry"].unique()
df_filtered_2 = df_C19_vaccination[df_C19_vaccination["TargetGroup"].isin(["ALL", "Age<18", "Age0_4", "Age5_9", "Age10_14", "Age15_17"])]
df_filtered_2 = df_filtered_2[df_filtered_2["Region"] == df_filtered_2["ReportingCountry"]]

exception_nations = ["CY", "DK", "IS", "LI", "LU", "NL", "PL", "PT", "SE", "SK"]

vaxPerCountry = pd.DataFrame({"ReportingCountry": [], "Procent": []})

for nation in nations:

    if nation in exception_nations:
        nation_df = df_filtered_2[df_filtered_2["ReportingCountry"] == nation]
    else:
        nation_df = df_filtered_2[df_filtered_2["ReportingCountry"] == nation]
        nation_df = nation_df[nation_df["TargetGroup"].isin(["ALL", "Age<18"])]
        
    try:
        First_dose_refused_percent = ((nation_df["Population"].iloc[0] - nation_df["FirstDose"].sum()) / nation_df["Population"].iloc[0]) * 100
    except:
        print(nation)
    vaxPerCountry.loc[len(vaxPerCountry)] = [nation, First_dose_refused_percent]

print(vaxPerCountry.sort_values("Procent", ascending=False).to_string(index=False))

#hejhehj

In [ ]:
#Cleaning
"""print(df_C19_hospital_ICU["indicator"].unique())
print(df_C19_hospital_ICU["source"].unique())
print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="External_Github"].unique())
print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="TESSy COVID-19 combined sources"].unique())
print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="TESSy NCOVAGGR"].unique())

print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="TESSy RESPISEVERE"].unique())
print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="TESSy RESPISURV"].unique())
print(df_C19_hospital_ICU["country"][df_C19_hospital_ICU["source"]=="TESSy RESPISEVERE"].unique())

print(df_C19_hospital_ICU["source"][df_C19_hospital_ICU["country"]=="Slovenia"].unique())




print(df_C19_hospital_ICU["url"].unique())"""

df_C19_hospital_ICU=df_C19_hospital_ICU.drop("url",axis=1)

print(df_C19_hospital_ICU[df_C19_hospital_ICU.isnull().any(axis=1)])



In [ ]:
#question 5 



cntrys = df_C19_hospital_ICU["country"].unique()
for land in länder:
    print(land, df_C19_hospital_ICU[df_C19_hospital_ICU["country"]==land]["indicator"].unique(),"\n")

"""df_C19_hospital_ICU_weekly = df_C19_hospital_ICU.copy()
df_C19_hospital_ICU_weekly = df_C19_hospital_ICU_weekly[df_C19_hospital_ICU_weekly["indicator"].isin(["Weekly new hospital admissions per 100k","Weekly new ICU admissions per 100k"])]
print(df_C19_hospital_ICU_weekly)"""





In [ ]:
#question 6

df_filtered_3 = df_C19_vaccination[df_C19_vaccination["TargetGroup"].isin(["Age<18", "Age0_4", "Age5_9", "Age10_14", "Age15_17"])]
df_filtered_3 = df_filtered_3[df_filtered_3["Region"] == df_filtered_3["ReportingCountry"]]

exception_nations = ["CY", "DK", "IS", "LI", "LU", "NL", "PL", "PT", "SE", "SK"]

vaxPerCountry = pd.DataFrame({"ReportingCountry": [], "Procent": []})

for nation in nations:

    if nation in exception_nations:
        nation_df_2 = df_filtered_3[df_filtered_3["ReportingCountry"] == nation]
    else:
        nation_df_2 = df_filtered_3[df_filtered_3["ReportingCountry"] == nation]
        nation_df_2 = nation_df_2[nation_df_2["TargetGroup"].isin(["Age<18"])]
        
    try:
        First_dose_percent = (nation_df_2["FirstDose"].sum() / nation_df_2["Population"].iloc[0]) * 100
    except:
        print(nation)
    vaxPerCountry.loc[len(vaxPerCountry)] = [nation, First_dose_percent]

print(vaxPerCountry.sort_values("Procent", ascending=False).to_string(index=False))

#he

# 3.7

In [ ]:
df_filtered_4 = df_C19_vaccination[df_C19_vaccination["TargetGroup"].isin(["Age60_69", "Age70_79", "Age80+", "1_Age60+"])]
df_filtered_4 = df_filtered_4[df_filtered_4["Region"] == df_filtered_4["ReportingCountry"]]

exception_nations = ["DE"]

vaxPerCountry_old = pd.DataFrame({"ReportingCountry": [], "Procent": []})

for nation in nations:

    if nation not in exception_nations:
        nation_df_3 = df_filtered_4[df_filtered_4["ReportingCountry"] == nation]
    else:
        nation_df_3 = df_filtered_4[df_filtered_4["ReportingCountry"] == nation]
        nation_df_3 = nation_df_3[nation_df_3["TargetGroup"].isin(["Age<18"])]
        
    try:
        First_dose_percent = (nation_df_3["FirstDose"].sum() / nation_df_3["Population"].iloc[0]) * 100
    except:
        print(nation)
    vaxPerCountry_old.loc[len(vaxPerCountry_old)] = [nation, First_dose_percent]

print(vaxPerCountry_old.sort_values("Procent", ascending=False).to_string(index=False))